# 예제 01. 과적합을 코드로 확인하기
빅데이터프로그래밍 · 9주차

## 목표
- 학습 데이터만 잘 맞히는 상태를 직접 만든다
- 학습 곡선 네 개를 함께 그려 읽는다
- 학습-검증 정확도 차이를 숫자로 확인한다

과적합은 이야기로 들으면 애매하지만, 곡선으로 보면 분명합니다.

**런타임 > 런타임 유형 변경 > T4 GPU** 를 먼저 선택하세요.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


## 1. 과적합을 잘 만드는 조건
데이터를 일부러 적게 씁니다. 2,000장만 쓰면 모델이 금방 외웁니다.


In [ ]:
transform = transforms.ToTensor()
full_train = datasets.FashionMNIST("./data", train=True,  download=True, transform=transform)
test_set   = datasets.FashionMNIST("./data", train=False, download=True, transform=transform)

small_train = Subset(full_train, range(2000))       # 2,000장만

train_loader = DataLoader(small_train, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_set, batch_size=256, shuffle=False)

print("학습:", len(small_train), "/ 검증:", len(test_set))


## 2. 모델 — 규제 장치가 하나도 없습니다


In [ ]:
class PlainCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 256), nn.ReLU(),
            nn.Linear(256, n_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))


model = PlainCNN().to(device)
print("파라미터:", f"{sum(p.numel() for p in model.parameters()):,}개")
print("학습 데이터:", len(small_train), "장")
print("→ 파라미터가 데이터보다 훨씬 많습니다")


## 3. 학습 — 네 가지를 모두 기록합니다


In [ ]:
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

def measure(loader):
    model.eval()
    loss_sum = correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss_sum += loss_fn(out, y).item() * y.numel()
            correct += (out.argmax(dim=1) == y).sum().item()
            total += y.numel()
    return loss_sum / total, correct / total


history = []
for epoch in range(1, 31):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        loss = loss_fn(model(x), y)
        opt.zero_grad(); loss.backward(); opt.step()

    tr = measure(train_loader)
    va = measure(test_loader)
    history.append((*tr, *va))

    if epoch % 5 == 0 or epoch == 1:
        print(f"epoch {epoch:2d}  학습 loss {tr[0]:.4f} acc {tr[1]:.4f}  |  "
              f"검증 loss {va[0]:.4f} acc {va[1]:.4f}")


## 4. 학습 곡선 네 개
왼쪽 그래프에서 **검증 손실이 다시 올라가는 지점**이 과적합의 시작입니다.


In [ ]:
tr_l = [h[0] for h in history]; tr_a = [h[1] for h in history]
va_l = [h[2] for h in history]; va_a = [h[3] for h in history]
xs = range(1, len(history) + 1)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].plot(xs, tr_l, label="학습 손실")
ax[0].plot(xs, va_l, label="검증 손실")
ax[0].set_title("loss"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(xs, tr_a, label="학습 정확도")
ax[1].plot(xs, va_a, label="검증 정확도")
ax[1].set_title("accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()


In [ ]:
best_epoch = int(torch.tensor(va_l).argmin()) + 1
print(f"검증 손실이 가장 낮았던 epoch: {best_epoch}  (loss {min(va_l):.4f})")
print(f"마지막 epoch 검증 손실      : {va_l[-1]:.4f}")
print(f"→ {best_epoch} epoch 이후로는 학습을 계속해도 나빠졌습니다")


## 5. 차이를 숫자로


In [ ]:
import pandas as pd

gap = [a - b for a, b in zip(tr_a, va_a)]
df = pd.DataFrame({
    "epoch": list(xs),
    "학습 정확도": [round(v, 4) for v in tr_a],
    "검증 정확도": [round(v, 4) for v in va_a],
    "차이": [round(v, 4) for v in gap],
})
print(df.iloc[[0, 4, 9, 14, 19, 24, 29]].to_string(index=False))

print(f"\n최종 차이: {gap[-1]:.4f}")
print("→ 0.10 이상이면 과적합이 뚜렷합니다" if gap[-1] > 0.10 else "→ 아직 심하지 않습니다")


In [ ]:
plt.figure(figsize=(7, 3.5))
plt.plot(xs, gap)
plt.axhline(0.10, color="crimson", linestyle="--", label="0.10")
plt.title("학습 정확도 − 검증 정확도"); plt.xlabel("epoch"); plt.legend(); plt.grid(alpha=.3)
plt.show()


## 6. 데이터를 늘리면 어떻게 되나
가장 확실한 해결책은 데이터를 늘리는 것입니다. 늘릴 수 없을 때 쓰는 것이 다음 노트북들의 방법입니다.


In [ ]:
big_loader = DataLoader(Subset(full_train, range(20000)), batch_size=64, shuffle=True)

model2 = PlainCNN().to(device)
opt2 = torch.optim.Adam(model2.parameters(), lr=1e-3)

for epoch in range(10):
    model2.train()
    for x, y in big_loader:
        x, y = x.to(device), y.to(device)
        loss = loss_fn(model2(x), y)
        opt2.zero_grad(); loss.backward(); opt2.step()

model_backup = model
model = model2
tr2, va2 = measure(big_loader), measure(test_loader)
model = model_backup

print(f"2,000장  최종 차이: {gap[-1]:.4f}")
print(f"20,000장 최종 차이: {tr2[1] - va2[1]:.4f}")


## 직접 해보기
1. 학습 데이터를 500장으로 줄이면 차이가 얼마나 벌어지나요?
2. epoch을 50으로 늘리면 검증 손실은 어떻게 되나요?


In [ ]:
# 여기에 작성하세요
